# Filtered Search with ACORN

*"The 10 nearest images whose price is below 100"*: a filtered search returns the nearest neighbors among the vectors that satisfy a predicate.

The naive approach, **post-filtering**, searches as usual and drops the results that fail the predicate. It works when most vectors pass, and collapses when few do: the true neighbors that pass are rarely among the unfiltered candidates.

kANNolo implements **ACORN** on its HNSW indexes: the filter is applied *during* the graph traversal.
- **ACORN-1** expands the two-hop neighbors of filtered-out nodes on the fly and needs no preprocessing.
- **ACORN-γ** precomputes expanded neighbor lists once (`build_acorn_gamma`) and keeps recall high on very selective filters.

This notebook compares the three on the LAION-small-CLIP filtered benchmark (100K CLIP embeddings, 512 dimensions, range predicates on a float attribute). Its test queries come with their exact filtered neighbors.

## Imports

In [ ]:
import importlib.metadata as md

print("kannolo", md.version("kannolo"))

In [ ]:
import sys
import time
from pathlib import Path

import numpy as np
import kannolo

# Download, loading and predicate helpers shared with scripts/acorn_benchmark.py
sys.path.insert(0, str(Path("..") / "scripts"))
import acorn_benchmark as ab

## Download and Load the Dataset

In [ ]:
dataset = "laion-small-clip"   # or "random_ints", "random_keywords" (2048-dimensional)
data_dir = Path("acorn_data")  # where the dataset is downloaded and extracted

info = ab.DATASETS[dataset]
print(info["description"])
dataset_dir = ab.download_and_extract(info["url"], data_dir)
vectors, payloads, tests = ab.load_dataset(dataset_dir)
n, dim = vectors.shape
print(f"{n} vectors of dimension {dim}, {len(tests)} test queries")

Each test query has a predicate (Qdrant's filter syntax) and its exact top neighbors among the vectors that satisfy it. Here each predicate is evaluated once and turned into a boolean mask over the collection. The searches below read the masks.

In [ ]:
k = 10
n_queries = 500  # test queries to use

queries = np.array([t["query"] for t in tests[:n_queries]], dtype=np.float32)
groundtruth = [t["closest_ids"][:k] for t in tests[:n_queries]]

# One row per query: the fastest way to pass one filter per query
masks = np.zeros((n_queries, n), dtype=bool)
for q, t in enumerate(tests[:n_queries]):
    predicate = ab.build_predicate(t["conditions"], payloads)
    masks[q] = np.fromiter((predicate(i) for i in range(n)), dtype=bool, count=n)
selectivity = masks.mean(axis=1)

print("Example predicate:", tests[0]["conditions"])
print(f"Selectivity (fraction of the collection a query may return): "
      f"min {selectivity.min():.2%}, median {np.median(selectivity):.2%}, max {selectivity.max():.2%}")

## Build the HNSW Index

In [ ]:
start = time.time()
index = kannolo.build(
    vectors,
    metric=info["metric"],                # inner product: cosine on normalized CLIP vectors
    encoder=kannolo.Plain("f16"),
    index=kannolo.HNSW(m=32, ef_construction=200),
)
print(f"Built in {time.time() - start:.1f} s: {index}")

All three methods search the whole batch at once. The 2-D mask array holds one filter per query.


In [ ]:
def recall(ids):
    return np.mean([len(set(r[r >= 0]) & set(g)) / len(g) for r, g in zip(ids, groundtruth)])


def post_filter(ef):
    """Search ef unfiltered candidates, keep those that pass, take the first k."""
    _, cand = index.search(queries, ef, ef_search=ef)
    out = np.full((len(queries), k), -1)
    for q, (row, mask) in enumerate(zip(cand, masks)):
        passing = row[(row >= 0) & mask[np.maximum(row, 0)]][:k]
        out[q, :len(passing)] = passing
    return out


def acorn(ef, gamma):
    _, ids = index.search(queries, k, ef_search=ef, filter=masks, acorn_gamma=gamma)
    return ids


def timed(fn, *args):
    start = time.time()
    ids = fn(*args)
    return ids, len(queries) / (time.time() - start)

In [ ]:
# ACORN-γ's expanded neighbor lists, built once (gamma of 2-4 is a good start)
start = time.time()
index.build_acorn_gamma(3)
print(f"ACORN-γ lists built in {time.time() - start:.1f} s")

ef_values = [20, 50, 100, 200, 400]
results = {}
print(f"{'ef_search':>9}  {'post-filter':>20}  {'ACORN-1':>20}  {'ACORN-γ':>20}")
for ef in ef_values:
    row = {}
    for name, fn, args in (("post-filter", post_filter, (ef,)),
                           ("ACORN-1", acorn, (ef, False)),
                           ("ACORN-γ", acorn, (ef, True))):
        ids, qps = timed(fn, *args)
        row[name] = (ids, recall(ids), qps)
    results[ef] = row
    print(f"{ef:>9}  " + "  ".join(f"{r:>8.3f} @ {q:>7.0f} QPS" for _, r, q in row.values()))

## Recall by Selectivity

Post-filtering loses most of its recall on selective filters: the true neighbors that pass are rarely among the unfiltered candidates. ACORN keeps searching the part of the graph that passes.

In [ ]:
ef = 100
buckets = [(0, 0.01, "< 1%"), (0.01, 0.1, "1-10%"), (0.1, 1.01, "> 10%")]
print(f"ef_search={ef}")
print(f"{'selectivity':>11}  {'queries':>7}  {'post-filter':>11}  {'ACORN-1':>8}  {'ACORN-γ':>8}")
for lo, hi, label in buckets:
    sel = (selectivity >= lo) & (selectivity < hi)
    if not sel.any():
        continue
    recalls = []
    for name in ("post-filter", "ACORN-1", "ACORN-γ"):
        ids = results[ef][name][0][sel]
        gt = [g for g, s in zip(groundtruth, sel) if s]
        recalls.append(np.mean([len(set(r[r >= 0]) & set(g)) / len(g) for r, g in zip(ids, gt)]))
    print(f"{label:>11}  {sel.sum():>7}  " + "  ".join(f"{r:>{w}.3f}" for r, w in zip(recalls, (11, 8, 8))))

## Notes

- A filter can be a boolean mask over the collection, an array of allowed ids, a callable `id -> bool`, or one of these per query: a list, or a 2-D boolean array with one row per query (as above). A callable is evaluated in Python for every id before the search, so use masks when you filter repeatedly. With masks, batch filtered search runs on every core (`num_threads=0`, the default).
- The lists built by `index.build_acorn_gamma(gamma)` stay in memory. `save` does not write them, so call it again after `load`.
- Filtered search applies to single-stage HNSW indexes, with any representation, encoder and graph layout.